# 💲💲💲 <span style="color: white; background-color: Purple"><b> Processo de Carga da Base de Histórico Salarial </b></span></p>

🧩 <span style="color: MediumSlateBlue"><b> 1- Carrega a base de controle de processos: </b></span></p>
- O script abre o arquivo PROCESSOS.xlsx
- E registra:
    - ID do processo  
    - Timestamp  
    - Etapa executada
- Isso permite rastrear o histórico de execuções (Etapa 0 → Etapa 1 → Etapa 2)

📁 <span style="color: MediumSlateBlue"><b> 2- Busca automaticamente os arquivos do Histórico Salarial: </b></span></p>
- Na pasta C:\\Users\\rodrigo.bernandes\\Downloads
- O script identifica arquivos cujo nome começa com REL HISTORICO SALARIAL
- Valida se existem arquivos e organiza a lista de processamento

🧹 <span style="color: MediumSlateBlue"><b> 3- Faz o tratamento e padronização completa da base: </b></span></p>
- Para cada arquivo encontrado, o script lê o Excel original
- Com pandas + openpyxl aplica o mapeamento completo das colunas
- Convertendo para padrão estabelicido conforme o dicionário
- Padroniza a ordem das colunas
- De acordo com o dicionário de mapeamento

📊 <span style="color: MediumSlateBlue"><b> 4- Gera a planilha final formatada: </b></span></p>
- O script cria HISTORICO SALARIAL.xlsx
- Com:
    - Sheet única  
    - Tabela estruturada no Excel  
    - Estilo TableStyleLight13  
    - Dados limpos e organizados
- A planilha final é própria para uso em RH

📦 <span style="color: MediumSlateBlue"><b> 5- Move arquivos processados para diretório histórico: </b></span></p>
- Após processar cada arquivo, ele é movido para X:\\Gestão de Pessoas\\Analytics\\03 - Bases\\2. ARQUIVOS MOVIDOS
- Isso mantém:
    - A área de Downloads limpa
    - Histórico organizado
    - Rastreabilidade dos arquivos brutos

🧾 <span style="color: MediumSlateBlue"><b> 6- Registra Etapa 2 e finaliza o controle de processos: </b></span></p>
- No final registra a conclusão no PROCESSOS.xlsx  
- Calcula o tempo total de execução  
- Exibe um resumo limpo e organizado no terminal

# Importação das Bibliotecas

In [1]:
import os
import pandas as pd
import pyautogui
import shutil
import time
from datetime import datetime, date
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo

# Carregando Base de Controle de Processos

In [2]:
id = 10

path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'
registros_processos = pd.read_excel(path_registros_processos, sheet_name="REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0
tempo_0 = [id, datetime.today(), 0]
ws_p.append(tempo_0)
wb_p.save(path_registros_processos)

# Variáveis com os Diretórios

In [3]:
# Diretório onde são salvos os arquivos extraídos
diretorio_buscar = r'C:\Users\rodrigo.bernandes\Downloads'

# Diretório onde são movidos os arquivos após a geração da base tratada
diretorio_mover = r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'

# Diretório onde será salvo o arquivo final
caminho_arquivo_final = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\HISTORICO SALARIAL.xlsx'

# Dataframe com o nome dos arquivos e data de extração
lista_arquivos = os.listdir(diretorio_buscar)
df_arquivos = pd.DataFrame(lista_arquivos, columns=['arquivo'])

# Filtrando apenas os arquivos que começam com 'REL HISTORICO SALARIAL'
df_arquivos = df_arquivos[df_arquivos['arquivo'].str.startswith('REL HISTORICO SALARIAL')].copy()

# Controle de atualização de processo: Etapa 1
tempo_1 = [id, datetime.today(), 1]
ws_p.append(tempo_1)
wb_p.save(path_registros_processos)

# Processo de Carga

In [4]:
# Dicionário de mapeamento para renomear as colunas
colunas_mapeamento = {
    'COD_EMP': 'cod_empresa',
    'REGISTRO': 'registro',
    'NOME': 'nome',
    'DIRETORIA': 'diretoria',
    'DIRET.DESC': 'desc_diretoria',
    'DEPTO.': 'departamento',
    'SETOR': 'cod_setor',
    'SETOR DESC': 'setor',
    'SECAO': 'cod_secao',
    'SECAO DESC': 'secao',
    'C.CUSTO': 'cod_ccusto',
    'CCUSTO DES': 'centro_custo',
    'MES/ANO': 'mes_ano_reajuste',
    'COL.%': 'aliquota',
    'IND.%': 'indice',
    'TIPO SAL.': 'tipo_sal',
    'MOTIVO COL': 'motivo_ajuste',
    'MOTIVO IND': 'motivo_ind',
    'CARGO': 'cod_cargo',
    'CARGO DESC': 'cargo'
}

for a in df_arquivos['arquivo'].tolist():

    arquivo = os.path.join(diretorio_buscar, a)

    historico_salarial = pd.read_excel(arquivo)

    # Seleciona apenas as colunas desejadas e as renomeia conforme o dicionário
    colunas_existentes = [col for col in colunas_mapeamento.keys() if col in historico_salarial.columns]
    historico_salarial = historico_salarial[colunas_existentes].rename(columns=colunas_mapeamento)

    wb = Workbook()
    ws = wb.active
    ws.title = "HISTORICO_SALARIAL"

    for r in dataframe_to_rows(historico_salarial, index=False, header=True):
        ws.append(r)

    tabela_historico_salarial = Table(displayName="HISTORICO_SALARIAL", ref=ws.dimensions)
    
    estilo_tabela = TableStyleInfo(
        name="TableStyleLight13", 
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )

    tabela_historico_salarial.tableStyleInfo = estilo_tabela

    ws.add_table(tabela_historico_salarial)

    wb.save(caminho_arquivo_final)  
                                       
    destino_tratados = os.path.join(diretorio_mover, a)
    
    shutil.move(arquivo, destino_tratados)
    
# Controle de atualização de processo: Etapa 2
tempo_2 = [id, datetime.today(), 2]
ws_p.append(tempo_2)
wb_p.save(path_registros_processos)

c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Atualizando o Arquivo Excel Controle HC e Atestados

In [5]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('HISTORICO_SALARIAL!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'f5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Resumo de Finalização do Processo

In [6]:
print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_2[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:00:29.517237

----------------------------------------------------------------------------------------------------
